In [13]:
import ast
import pandas as pd

countries_clean = pd.read_csv("../raw/countries_raw.csv")

dups = countries_clean.duplicated().sum()
print(f"Duplicate rows: {dups}")

dups_id = countries_clean.duplicated(subset=["id"]).sum()
print(f"Duplicate id: {dups_id}")

print(f"\nShape: {countries_clean.shape}")

Duplicate rows: 0
Duplicate id: 0

Shape: (296, 10)


In [14]:
print(countries_clean["region"].iloc[0])
print(countries_clean["incomeLevel"].iloc[0])
print(countries_clean["lendingType"].iloc[0])

{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Latin America & Caribbean '}
{'id': 'HIC', 'iso2code': 'XD', 'value': 'High income'}
{'id': 'LNX', 'iso2code': 'XX', 'value': 'Not classified'}


In [15]:
def parse_dict_col(val, key="value"):
    try:
        return ast.literal_eval(val).get(key, None)
    except:
        return None

# Parse all nested dict columns
for col in ["region", "adminregion", "incomeLevel", "lendingType"]:
    countries_clean[col] = countries_clean[col].apply(parse_dict_col)

# Rename all columns to snake_case
countries_clean.rename(columns={
    "id":          "country_code_iso3",
    "iso2Code":    "country_code_iso2",
    "name":        "country_name",
    "incomeLevel": "income_level",
    "lendingType": "lending_type",
    "adminregion": "admin_region",
    "capitalCity": "capital_city"
}, inplace=True)

countries_clean.head(3).to_string()
print(f"\nUnique income levels: {countries_clean['income_level'].unique()}")
print(f"Unique lending types: {countries_clean['lending_type'].unique()}")


Unique income levels: ['High income' 'Aggregates' 'Low income' 'Lower middle income'
 'Upper middle income' 'Not classified']
Unique lending types: ['Not classified' 'Aggregates' 'IDA' 'IBRD' 'Blend']


In [19]:
## aggregated ?? -> These are pre-aggregated statistical groups, not sovereign countries. 
#The World Bank uses them for high-level reports and dashboards.
countries_clean = countries_clean[countries_clean["income_level"] != "Aggregates"].reset_index(drop=True)

countries_clean = countries_clean[countries_clean["capital_city"].notna()].reset_index(drop=True)

nulls = countries_clean.isnull().sum()
print(f"Shape after cleaning: {countries_clean.shape}")
print(f"\nRemaining nulls:")
print(nulls[nulls > 0])

print(f"\nIncome level breakdown:")
print(countries_clean["income_level"].value_counts())

Shape after cleaning: (211, 10)

Remaining nulls:
longitude    3
latitude     3
dtype: int64

Income level breakdown:
income_level
High income            81
Upper middle income    54
Lower middle income    49
Low income             25
Not classified          2
Name: count, dtype: int64


In [17]:
countries_clean["country_code_iso2"] = countries_clean["country_code_iso2"].fillna("Unknown")

print(f"Final shape: {countries_clean.shape}")
print(f"\nRemaining nulls:")
print(countries_clean.isnull().sum()[countries_clean.isnull().sum() > 0])

print(f"\nLending type breakdown:")
print(countries_clean["lending_type"].value_counts())

Final shape: (211, 10)

Remaining nulls:
longitude    3
latitude     3
dtype: int64

Lending type breakdown:
lending_type
IBRD              67
Not classified    66
IDA               59
Blend             19
Name: count, dtype: int64


In [18]:
countries_clean.to_csv("../cleaned/countries_clean.csv", index=False)
print("Saved: ../cleaned/countries_clean.csv")

Saved: ../cleaned/countries_clean.csv
